## Download JPEGs and Calculate Extents

Prepare all photos in a selected subset for georeferencing.

- Download original JPEGs
- Read image dimensions and DPI
- Calculate approximate horizontal ground extents
- Save preparation results to a separate tracker GeoPackage

Existing valid JPEGs are reused rather than downloaded again.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm
import shutil

from single_photo_workflow import (
    download_photo,
    get_image_metadata,
    calculate_horizontal_extent,
    update_tracker,
    ground_to_mercator_extent
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

### Load subset table and make paths

In [ ]:
SUBSET_PATH = Path(
    "data/historic_airphoto_index/subsets/"
    "subset_20000_carver_37_38_40_51_57_63.gpkg"
)

LAYER = "photos"

# Create a separate GeoPackage for tracking processing progress
TRACKER_PATH = SUBSET_PATH.with_name(
    f"{SUBSET_PATH.stem}_tracker.gpkg"
)

# define subfolder for downloading images, using the geopackage name
DOWNLOAD_DIR = Path("data/downloads") / SUBSET_PATH.stem

### Make tracker geopackage and download directory

In [ ]:
# Create tracker gpkg, ONLY if it doesn't already exist
# This is a copy of the subset table, which is preserved blank for input. 

if not TRACKER_PATH.exists():
    shutil.copy2(SUBSET_PATH, TRACKER_PATH)
    print(f"Created tracker: {TRACKER_PATH.name}")
else:
    print(f"Using existing tracker: {TRACKER_PATH.name}")

In [ ]:
# Make download directory if it doesn't already exist
# exist_ok makes it succeed silently and not do anything if it already exists
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

### Load tracker, select unfinished photos

In [ ]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

In [ ]:
tracker_gdf

In [ ]:
# Set to True to re-prepare all photos, regardless of status.
# does not re-download the photo if it exists. to do that, delete the files
REPREPARE_ALL = False

if REPREPARE_ALL:
    photos_gdf = tracker_gdf.copy()
else:
    photos_gdf = tracker_gdf.loc[
        ~tracker_gdf["status"].isin(["prepared", "georeferenced"])
    ].copy()

print(f"Total photos in subset: {len(tracker_gdf)}")
print(f"Photos to prepare: {len(photos_gdf)}")
print(f"Download directory: {DOWNLOAD_DIR}")

display(photos_gdf.head())

### Download JPEG, read metadata, calculate extent

In [ ]:
# create an empty list to collect results for each photo. 
results = []

# loop through photos_gdf, processing one photo (row) at a time. 
for _, photo in tqdm(
    photos_gdf.iterrows(),
    total=len(photos_gdf),
    desc="Preparing photos"
):
    # get the unique but friendly photo identifier
    photo_id = photo["cty_year_id"]

    # and use it to construct the download path
    image_path = DOWNLOAD_DIR / f"{photo_id}_original.jpg"

    # Create a dictionary to store this photo's processing results.
    # Default values are replaced as each step succeeds.
    result = {
        "source_id": photo["source_id"],
        "width_px": None,
        "dpi_x": None,
        "ground_extent": None,
        "h_extent": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # download JPEG or reuse if already downloaded
        image_path = download_photo(
            url=photo["url"],
            output_path=image_path,
        )

        # record download success and date

        result["date_prepared"] = datetime.now(
            timezone.utc
        ).isoformat()

        # read metadata from the downloaded image
        metadata = get_image_metadata(image_path)

        # update that dictionary
        result.update(metadata)

        # estimate the photos horizontal ground coverage in meters
        result["ground_extent"] = calculate_horizontal_extent(
            width_px=metadata["width_px"],
            dpi_x=metadata["dpi_x"],
            scale=photo["scale"],
        )

        # add web mercator distortion to ground width, based on latitude
        result["h_extent"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_extent"],
            latitude = photo["latitude"],
        )

    except Exception as e:
        # If any step fails, record the error and continue
        result["error"] = str(e)

    # Save this photo's result dict to the list of dicts
    results.append(result)

In [ ]:
# convert list of results dicts into a dataframe
results_df = pd.DataFrame(results)

In [ ]:
# examine
results_df

In [ ]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

In [ ]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf